---
## Assignment 1.3: Visualizing Distributions

This exercise asks you to recreate several classic plot types from DAOST Chapter 2 using your own crime data — putting visualization theory into practice.

*Draws from*: Week 3, Exercises 5.2 and 5.3.

> **Part A — Jitter plot**
> * Pick one of your Personal Focus Crimes and a suitable time interval (somewhere between a month and 6 months, depending on how common the crime is). Create a jitter plot of the incident times during a single hour (e.g. 13:00–14:00): let time run along the $x$-axis and add vertical jitter.
> * What does the jitter plot reveal about how times are recorded in the dataset? Are incidents clustered at certain minutes (on the hour, half hour, etc.)? What does this tell you about the precision of the data?
>
> **Part B — Probability plot**
> * Using the same geographic data from Part B, create a probability plot (QQ plot) for the latitude distribution of each of your two crime types. (`scipy.stats.probplot` is your friend here.)
> * What reference distribution are you comparing against? What would it mean if the points fell exactly on the straight line? Where does the distribution deviate from normal, and what does that deviation tell you about the geography of crime in SF?
>
> **Part C — Box plots of time-of-day**
> * For each of your Personal Focus Crimes, extract the time-of-day of every incident.
> * Create box plots showing the time-of-day distribution for all your Personal Focus Crimes side by side.
> * What patterns do you see? Are there crimes that happen mostly at night? Mostly during business hours? For crimes that peak late at night, does the box plot handle the wrap-around at midnight well? What goes wrong?
> * Above, feel free to use alternatives to box plots — violin plots, swarm plots, or raincloud plots — if you think they reveal more. If you do, briefly explain what the alternative shows that the box plot doesn't.

In [ ]:
# Load data
import pandas as pd
df = pd.read_csv('~/OneDrive/Desktop/sdav/exercises/data/merged_sfpd.csv')

personal_focus = [
    'larceny/theft',
    'non-criminal',
    'assault',
    'vehicle theft',
    'drug/narcotic',
    'vandalism',
    'warrants',
    'burglary',
    'suspicious occ'
]

df_pf = df[df['incident_category'].isin(personal_focus)].copy()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df_pf['incident_time'] = pd.to_datetime(df_pf['incident_time'], format='%H:%M:%S')
df_pf['hour'] = df_pf['incident_time'].dt.hour
df_pf['minute'] = df_pf['incident_time'].dt.minute
hour13 = df_pf[(df_pf['incident_date'] > '2025-06-01') & (df_pf['incident_date'] < '2025-12-31') & (df_pf['hour'] == 13) & (df_pf['incident_category'] == 'larceny/theft')]

plt.figure(figsize=(10, 2))
plt.scatter(hour13['minute'], [0.2]*len(hour13), 10, marker='x', color='black', linewidths=1)
plt.scatter(hour13['minute'], np.random.uniform(0, 0.15, len(hour13)), 10, marker='o', color='red', linewidths=1)
plt.yticks([])
plt.xlabel('Minute of the Hour')
plt.title('Incidents Occurring at 1 PM')
plt.show()

The jitter plot above shows us when, during the 13th hour of the day, that crimes are registered. Without being critical toward the data source it would seem that criminals are politely waiting for "nice" time-slots for getting caught. We see that most crimes occur at the whole hour and second most at the half hour. There is also a noticeable part at the first quarter, but funnily enough the third quarter is much less pronounced. Of course this is due to a "laziness" from the police officers as they note the time of the crime occurrence. What is less obvious is why there is a tendency to note the first quarter and not the third. Is there truly more crime in the first half hour or is there some hidden systematical error in the data at this point?

In [ ]:
from scipy.stats import probplot
crimes = ['vehicle theft', 'drug/narcotic']
df_pf = df_pf[df_pf['latitude'] < 80]

plt.figure(figsize=(12, 5))
ax = plt.subplot(1, 2, 1)
probplot(df_pf[df_pf['incident_category'] == crimes[0]]['latitude'], dist="norm", plot=plt)
plt.title(f'Q-Q Plot for Latitude {crimes[0]} crimes in SF')
plt.subplot(1, 2, 2, sharey=ax)
probplot(df_pf[df_pf['incident_category'] == crimes[1]]['latitude'], dist="norm", plot=plt)
plt.title(f'Q-Q Plot for Latitude of {crimes[1]} crimes in SF')
plt.show()

Here we are investigating the latitude distribution of two specific crimes; vehicle theft and drugs. The Q-Q plots above matches the distribution of latitude coordinates to a normal distribution. Had the blue lines followed the red, the data would have been perfectly normal. That is not the case. For the vehicle thefts in the left plot we see a high tail in the lower end of the plot. This means that the data is skewed to the left, compared to a normal distribution. In geographical terms, it means that a surplus of vehicle thefts happen in the southern regions of SF. Thinking back to part 1.2 of this assignment, that makes a lot of sense, since the most crime happens in the southern district. Logically, it makes sense since "downtown" SF is in the northern part of the city, meaning that the majority of cars are probably to the south.

Looking a bit to the right we see the latitude distribution of the drug/narcotics crimes. Now, these are quite different. Here there is a high region in the middle meaning that the distribution is rather hyper-normal. Again in geographical terms, it seems that the central regions of SF is where people a caught doing drugs. Putting this into perspective is a bit hard, without knowing the city. But from an objective standpoint one would assume that the the smaller "off-broadway" streets located far enough from downtown to be on spotlight, but close enough to still be in the city would be a good place for a party or perhaps a drug den.

In either case it seems that the downtown area is kept somewhat clean from these categories of crimes.
 

In [ ]:
data = df_pf.copy()

data['time_since_midnight'] = data['hour'] * 60 + data['minute']
df = {cat: data[data['incident_category'] == cat]['time_since_midnight'] for cat in personal_focus}

plt.violinplot(list(df.values()), positions=list(range(len(df))))
plt.xticks(list(range(len(df))), list(df.keys()), rotation=45)
plt.ylabel("Minutes since midnight")
plt.title("Distribution of Incident Times by Category")
plt.show()

In the plot you see a distribution of the time of day for each focus category of crime. For this task, we have chosen the slightly niche violin-plot, instead of the usual boxplot. This is done to avoid the otherwise big fallacy of visualizing days as a non-circular timeframe. For a regular boxplot we would usually see an average at the middle of the day, this is true for a crime, which mostly occurs in the evening / at night, or a crime which mostly occurs during mid day. For the violinplot we instead see the explicit distribution, allowing the viewer to decide when a crime mostly occurs. E.g. in the case we see that larceny/theft is a crime most occurring in the evening/night hours, but dies out later during the night. On the opposite side we see the drug crimes mostly happen during the day/evening, which is opposite to what one might believe. Unsurprisingly though, burglary can be seen to occur somewhat evenly throughout all 24 hours.